T4 GPUが利用可能かどうか確認

In [ ]:
# 1. GPU (T4等) が利用可能か自動チェック
import sys
import torch

if torch.cuda.is_available():
    device = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✅ GPUが有効です: {gpu_name}")
else:
    # CPUの場合は警告を出す
    print("⚠️ 警告: GPUが有効になっていません！")
    print("Colabの上部メニュー [ランタイム] -> [ランタイムのタイプを変更] から『T4 GPU』を選択してください。")

In [ ]:
# ===========================
# 1. 必要なライブラリの準備
# ===========================
!pip install -q diffusers transformers accelerate torch

import torch
# SD 1.5 用のパイプライン
from diffusers import StableDiffusionPipeline

セル1：モデルの読み込み（最初の1回だけ実行）

In [ ]:
# ---------------
# 注意：GPUを使う
# ---------------
import warnings
warnings.filterwarnings("ignore")

import torch
from diffusers import StableDiffusionPipeline

# モデルをGPUにロード（最初だけファイル取得が行われます）
model_id = "runwayml/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    dtype=torch.float16
).to("cuda")

print("モデルの準備が完了しました！")

セル2：画像生成（プロンプトやシードを変えて何度でも実行）

市民の夢をかなえた便利で、多くの商店が集まった駅前のカラーイメージ画像

In [ ]:
# プロンプトやシード値を変更
prompt = (
"A colorful image of the street in Japan, where the convenience fulfilled the citizens' dreams and many shops gathered"
"some people are walking around the shopping street."
"A stylish shopping street with Poplar tree street trees along the edge of the sidewalk"
)
negative_prompt = "night, no sunny, anime, blurry"

# シード値を変えると別の構図になります（例: 42, 101, 777 など）
new_generator = torch.Generator("cuda").manual_seed(700)

# 画像生成
image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    width=768,
    height=512,
    num_inference_steps=30,
    guidance_scale=7.5, # 7.0 〜 8.0：最適のバランス、　1.0 〜 4.0：夢がある
    generator= new_generator
).images[0]

# 表示と保存
image.save("night_street_new.png")
display(image)

テキストだけで指示すると、Stable Diffusion（海外の画像を中心に学習したモデル）は欧米風の街並みや古い映画のような極端な絵を描いてしまい、「日本のリアルな生活道路」からかけ離れてしまう。

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import torch
from PIL import Image
# img2img 専用のパイプラインをインポート
from diffusers import StableDiffusionImg2ImgPipeline

# 1. モデルの読み込み（SD v1.5 の img2img版）
model_id = "runwayml/stable-diffusion-v1-5"
pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    model_id,
    dtype=torch.float16
).to("cuda")

# 2. 参考となる元画像を読み込む（サイズを整える）
input_image_path = "input_street.jpg"  # アップロードしたファイル名
init_image = Image.open(input_image_path).convert("RGB")
init_image = init_image.resize((768, 512))  # 幅768、高さ512にリサイズ

print("元画像を読み込みました:")
display(init_image)

# 3. 「夜にする・暗くする」プロンプト
prompt = (
    "A realistic documentary photo of this Japanese residential road transformed into pitch black midnight, "
    "extremely dark night, very poorly light, only one dim distant flickering streetlight, "
    "heavy dark shadows, hazardous dangerous night atmosphere, 4k high quality"
)

negative_prompt = (
    "daytime, bright, sunny, cheerful, neon lights, anime, painting, blurry, distorted"
)

# 4. 画像生成（元画像を参照して描画）
generator = torch.Generator("cuda").manual_seed(42)

image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=init_image,      # ★ここで参考画像を渡す
    strength=0.65,         # ★元画像をどのくらい変化させるか（超重要パラメータ）
    guidance_scale=8.0,
    generator=generator
).images[0]

"""
# 5. 保存と表示
image.save("night_transformed.png")
print("変換が完了しました！")
display(image)
"""